# Adaptive Reranking with Logistic Regression

This notebook walks through the full pipeline for **adaptive re-ranking** using a **logistic regression** model that decides, for each query, whether re-ranking should be applied.

We assume that:
- Phase 1 (VPR) predictions have already been computed and saved as `.txt` files.
- Phase 2 (image matching) inlier statistics have already been computed and saved as `.torch` files.
- The directory structure and formats follow the rest of this project.

We will:
1. Load per-query predictions and inlier statistics.
2. Define what a **hard query** is.
3. Build features and labels.
4. Train a logistic regression model.
5. Evaluate an **adaptive reranking** strategy and compute **cost savings**.



In [ ]:
import os
from glob import glob
from pathlib import Path

import numpy as np
import torch
import joblib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

from util import get_list_distances_from_preds, read_file_preds


# --- User parameters ---
# Point these to one specific experiment (one VPR method + one matcher + one dataset split)
PREDS_DIR = Path("logs/my_experiment/<timestamp>")  # Phase 1 outputs (.txt)
INLIERS_DIR = PREDS_DIR.parent / f"{PREDS_DIR.name}_superpoint-lg"  # or your matcher name
NUM_PREDS = 20
POSITIVE_DIST_THRESHOLD = 25.0  # meters
MODEL_PATH = Path("logistic_regressor.joblib")  # optional: load a pre-trained model

print("Preds dir:", PREDS_DIR)
print("Inliers dir:", INLIERS_DIR)



In [ ]:
def collect_per_query_stats(preds_dir: Path, inliers_dir: Path, num_preds: int, positive_dist_threshold: float):
    txt_files = glob(str(preds_dir / "*.txt"))
    txt_files.sort(key=lambda x: int(Path(x).stem))

    records = []

    for txt_file_query in txt_files:
        geo_dists = torch.tensor(get_list_distances_from_preds(txt_file_query))[:num_preds]
        if geo_dists.numel() == 0:
            continue

        torch_file_query = inliers_dir / (Path(txt_file_query).name.replace("txt", "torch"))
        if not torch_file_query.exists():
            continue

        query_results = torch.load(torch_file_query, weights_only=False)
        if len(query_results) == 0:
            continue

        num_considered = min(num_preds, len(query_results))
        inliers = torch.zeros(num_considered, dtype=torch.float32)
        for i in range(num_considered):
            inliers[i] = query_results[i]["num_inliers"]

        # Baseline (before reranking)
        baseline_top1_dist = float(geo_dists[0])
        baseline_correct = baseline_top1_dist <= positive_dist_threshold

        # After reranking (by inliers)
        inliers_sorted, indices = torch.sort(inliers, descending=True)
        geo_dists_reranked = geo_dists[:num_considered][indices]
        rerank_top1_dist = float(geo_dists_reranked[0])
        rerank_correct = rerank_top1_dist <= positive_dist_threshold

        record = {
            "query_id": int(Path(txt_file_query).stem),
            "top1_inliers": float(inliers[0]),
            "mean_inliers": float(inliers.mean()),
            "max_inliers": float(inliers.max()),
            "std_inliers": float(inliers.std()) if num_considered > 1 else 0.0,
            "baseline_correct": baseline_correct,
            "rerank_correct": rerank_correct,
        }
        records.append(record)

    return records


records = collect_per_query_stats(PREDS_DIR, INLIERS_DIR, NUM_PREDS, POSITIVE_DIST_THRESHOLD)
len(records)


In [ ]:
import pandas as pd

# Convert to DataFrame for easier analysis
_df = pd.DataFrame.from_records(records)
print(_df.head())
print("\nBaseline R@1:", _df["baseline_correct"].mean() * 100.0)
print("Reranked R@1:", _df["rerank_correct"].mean() * 100.0)

plt.figure(figsize=(6, 4))
plt.hist(_df["top1_inliers"], bins=40, alpha=0.7)
plt.xlabel("Top-1 inliers (before reranking)")
plt.ylabel("#queries")
plt.title("Distribution of top-1 inliers")
plt.show()


In [ ]:
# Define labels for hard queries
# Option 1: hard if baseline is wrong but reranking fixes it (benefit_from_rerank)
_df["hard_benefit"] = (~_df["baseline_correct"] & _df["rerank_correct"]).astype(int)

# Option 2: hard if baseline is wrong (baseline_incorrect)
_df["hard_baseline_incorrect"] = (~_df["baseline_correct"]).astype(int)

print("Class balance (benefit_from_rerank):", _df["hard_benefit"].value_counts().to_dict())
print("Class balance (baseline_incorrect):", _df["hard_baseline_incorrect"].value_counts().to_dict())


In [ ]:
from sklearn.model_selection import train_test_split

FEATURE_COLS = ["top1_inliers", "mean_inliers", "max_inliers", "std_inliers"]
TARGET_COL = "hard_benefit"  # you can switch to "hard_baseline_incorrect" if preferred

X = _df[FEATURE_COLS].values.astype(np.float32)
y = _df[TARGET_COL].values.astype(np.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(solver="lbfgs", max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)

print("Train accuracy:", accuracy_score(y_train, clf.predict(X_train)))
print("Val accuracy:", accuracy_score(y_val, clf.predict(X_val)))
print("Confusion matrix (val):")
print(confusion_matrix(y_val, clf.predict(X_val)))

# Optionally save the model
joblib.dump(clf, MODEL_PATH)
print("Saved logistic regression model to", MODEL_PATH)


In [ ]:
# Adaptive reranking using the trained logistic regression model

# If you prefer, you can reload from disk:
# clf = joblib.load(MODEL_PATH)

probs_hard = clf.predict_proba(X)[:, 1]
threshold_prob = 0.5  # you can tune this on the validation set

_df["prob_hard"] = probs_hard
_df["pred_hard"] = (probs_hard >= threshold_prob).astype(int)

# Strategy:
# - If pred_hard == 1: apply reranking (use rerank_correct)
# - If pred_hard == 0: keep baseline (use baseline_correct)

adaptive_correct = np.where(
    _df["pred_hard"] == 1,
    _df["rerank_correct"].values,
    _df["baseline_correct"].values,
)

always_rerank_correct = _df["rerank_correct"].values

print("Baseline R@1:", _df["baseline_correct"].mean() * 100.0)
print("Always rerank R@1:", always_rerank_correct.mean() * 100.0)
print("Adaptive (logistic) R@1:", adaptive_correct.mean() * 100.0)

# Cost: how many queries trigger reranking?
frac_reranked = _df["pred_hard"].mean()
print("Fraction of queries reranked (adaptive):", frac_reranked)
print("Cost saving vs always rerank:", (1.0 - frac_reranked) * 100.0, "%")
